# Cuaderno de apoyo — Fundamentos de Machine Learning

# PROGRAMA 7 — Hold-Out · DummyClassifier · SVC · Guardado de modelos

In [ ]:
import numpy as np
import pickle
from pathlib import Path
from sklearn import datasets
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

iris = datasets.load_iris()
X = iris.data
y = iris.target

print("Dimensiones:", X.shape)
print("Especies:", iris.target_names)

clases, cantidades = np.unique(y, return_counts=True)
print("Clases:", clases)
print("Cantidad por clase:", cantidades)

In [ ]:
X_training, X_test, y_training, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_training, y_training,
    test_size=0.2,
    random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Validación:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
standardizer = StandardScaler()

X_train_scaled = standardizer.fit_transform(X_train)
X_val_scaled = standardizer.transform(X_val)
X_test_scaled = standardizer.transform(X_test)

print("Train:", X_train_scaled.shape)
print("Validación:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

In [ ]:
dummy = DummyClassifier(strategy="prior", random_state=42)
dummy.fit(X_train_scaled, y_train)

print("Dummy - Exactitud en validación:", round(dummy.score(X_val_scaled, y_val) * 100, 2), "%")

predicciones_dummy = dummy.predict(X_val_scaled)
print("Predicciones:", predicciones_dummy)
print("Valores reales:", y_val)

In [ ]:
svc = SVC(C=0.5)
svc.fit(X_train_scaled, y_train)

predicciones_val = svc.predict(X_val_scaled)
exactitud_val = svc.score(X_val_scaled, y_val)

print("Predicciones de validación:", predicciones_val)
print("Etiquetas reales:", y_val)
print("Exactitud en validación:", round(exactitud_val * 100, 4), "%")

exactitud_test = svc.score(X_test_scaled, y_test)
print("Exactitud en test:", round(exactitud_test * 100, 4), "%")

In [ ]:
Path("models").mkdir(parents=True, exist_ok=True)

with open("models/modelo_iris.pickle", "wb") as archivo:
    pickle.dump({
        "modelo": svc,
        "estandarizacion": standardizer,
        "clases": iris.target_names
    }, archivo)

print("Archivo guardado: models/modelo_iris.pickle")

In [ ]:
with open("models/modelo_iris.pickle", "rb") as archivo:
    objetos = pickle.load(archivo)

modelo_cargado = objetos["modelo"]
escalador_cargado = objetos["estandarizacion"]
nombres_clases = objetos["clases"]

flor_nueva = np.array([[5.1, 3.5, 1.4, 0.2]])
flor_escalada = escalador_cargado.transform(flor_nueva)
clase_predicha = int(modelo_cargado.predict(flor_escalada)[0])

print("Clase predicha:", clase_predicha)
print("Especie:", nombres_clases[clase_predicha])

In [ ]:
largo_sepalo = float(input("Largo del sépalo: "))
ancho_sepalo = float(input("Ancho del sépalo: "))
largo_petalo = float(input("Largo del pétalo: "))
ancho_petalo = float(input("Ancho del pétalo: "))

flor_manual = np.array([[
    largo_sepalo,
    ancho_sepalo,
    largo_petalo,
    ancho_petalo
]])

flor_manual_escalada = escalador_cargado.transform(flor_manual)
clase_manual = int(modelo_cargado.predict(flor_manual_escalada)[0])

print("Especie predicha:", nombres_clases[clase_manual])

# PROGRAMA 8 — KFold

In [ ]:
from sklearn.model_selection import KFold

letras = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for numero_fold, (indices_train, indices_val) in enumerate(kf.split(letras), start=1):
    letras_train = [letras[i] for i in indices_train]
    letras_val = [letras[i] for i in indices_val]

    print("Fold:", numero_fold)
    print("Índices de entrenamiento:", indices_train)
    print("Índices de validación:", indices_val)
    print("Letras de entrenamiento:", letras_train)
    print("Letras de validación:", letras_val)
    print()

# PROGRAMA 9 — Hold-Out externo y validación cruzada interna

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

iris = datasets.load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

modelo = make_pipeline(
    StandardScaler(),
    SVC(C=0.5, gamma="auto")
)

print("Entrenamiento:", X_train.shape)
print("Test externo:", X_test.shape)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

resultados = cross_val_score(
    modelo,
    X_train,
    y_train,
    cv=kf,
    scoring="accuracy"
)

print("Exactitud por fold:", resultados)
print("Exactitud media:", round(resultados.mean() * 100, 4), "%")
print("Desviación estándar:", round(resultados.std() * 100, 4), "puntos porcentuales")

In [ ]:
modelo.fit(X_train, y_train)

predicciones_test = modelo.predict(X_test)
exactitud_test = modelo.score(X_test, y_test)

print("Predicciones:", predicciones_test)
print("Valores reales:", y_test)
print("Exactitud en test:", round(exactitud_test * 100, 4), "%")

In [ ]:
valores_c = [0.1, 0.5, 1, 10]

for valor_c in valores_c:
    candidato = make_pipeline(
        StandardScaler(),
        SVC(C=valor_c, gamma="auto")
    )

    puntuaciones = cross_val_score(
        candidato,
        X_train,
        y_train,
        cv=kf,
        scoring="accuracy"
    )

    print("C:", valor_c, "Media:", round(puntuaciones.mean(), 4))

# PROGRAMA 10 — Regresión con Diabetes · MAE, RMSE y MAPE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer

diabetes = datasets.load_diabetes()
X = diabetes.data
y = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Datos:", X.shape)
print("Entrenamiento:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
modelo_regresion = make_pipeline(
    StandardScaler(),
    SVR(C=10, gamma="scale", kernel="rbf")
)

def calcular_rmse(valores_reales, predicciones):
    return np.sqrt(mean_squared_error(valores_reales, predicciones))

def calcular_mape(valores_reales, predicciones):
    return np.mean(np.abs((valores_reales - predicciones) / valores_reales)) * 100

metricas = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": make_scorer(calcular_rmse, greater_is_better=False),
    "MAPE": make_scorer(calcular_mape, greater_is_better=False)
}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

resultados = cross_validate(
    modelo_regresion,
    X_train,
    y_train,
    cv=kf,
    scoring=metricas
)

print("MAE de cada fold:", -resultados["test_MAE"])
print("RMSE de cada fold:", -resultados["test_RMSE"])
print("MAPE de cada fold:", -resultados["test_MAPE"])

print("MAE promedio:", -resultados["test_MAE"].mean())
print("RMSE promedio:", -resultados["test_RMSE"].mean())
print("MAPE promedio (%):", -resultados["test_MAPE"].mean())

In [ ]:
modelo_regresion.fit(X_train, y_train)
predicciones = modelo_regresion.predict(X_test)

mae = mean_absolute_error(y_test, predicciones)
mse = mean_squared_error(y_test, predicciones)
rmse = np.sqrt(mse)
mape = calcular_mape(y_test, predicciones)

print("MAE:", round(mae, 4))
print("MSE:", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("MAPE (%):", round(mape, 4))
print("Valores reales:", np.round(y_test[:10], 1))
print("Predicciones:", np.round(predicciones[:10], 1))

In [ ]:
plt.scatter(y_test, predicciones)

limite_minimo = min(y_test.min(), predicciones.min())
limite_maximo = max(y_test.max(), predicciones.max())

plt.plot(
    [limite_minimo, limite_maximo],
    [limite_minimo, limite_maximo],
    "k--"
)

plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Regresión: valores reales frente a predicciones")
plt.show()

# PROGRAMA 11 — Clasificación · cross_val_predict · matriz de confusión

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

iris = datasets.load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

modelo_clasificacion = make_pipeline(
    StandardScaler(),
    SVC(C=0.1, gamma="auto", kernel="rbf")
)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

predicciones_cv = cross_val_predict(
    modelo_clasificacion,
    X_train,
    y_train,
    cv=cv
)

print("Predicciones:", predicciones_cv)
print("Clases reales:", y_train)

In [ ]:
accuracy = accuracy_score(y_train, predicciones_cv)
precision = precision_score(y_train, predicciones_cv, average="macro", zero_division=0)
recall = recall_score(y_train, predicciones_cv, average="macro", zero_division=0)
f1 = f1_score(y_train, predicciones_cv, average="macro", zero_division=0)

print("Accuracy:", round(accuracy * 100, 4), "%")
print("Precision:", round(precision * 100, 4), "%")
print("Recall:", round(recall * 100, 4), "%")
print("F1:", round(f1 * 100, 4), "%")

In [ ]:
matriz = confusion_matrix(y_train, predicciones_cv)
informe = classification_report(
    y_train,
    predicciones_cv,
    target_names=iris.target_names,
    zero_division=0
)

print("Matriz de confusión:")
print(matriz)
print("Informe de clasificación:")
print(informe)

In [ ]:
modelo_clasificacion.fit(X_train, y_train)

predicciones_test = modelo_clasificacion.predict(X_test)
exactitud_test = accuracy_score(y_test, predicciones_test)

print("Exactitud de test:", round(exactitud_test * 100, 4), "%")
print("Matriz de confusión de test:")
print(confusion_matrix(y_test, predicciones_test))
print("Informe de test:")
print(classification_report(
    y_test,
    predicciones_test,
    target_names=iris.target_names,
    zero_division=0
))

# Fórmulas en código con datos de práctica

In [ ]:
import numpy as np

valores_reales = np.array([100, 200, 300, 400])
valores_predichos = np.array([110, 180, 330, 390])

errores = valores_reales - valores_predichos
mae = np.mean(np.abs(errores))
mse = np.mean(errores ** 2)
rmse = np.sqrt(mse)
mape = np.mean(np.abs(errores / valores_reales)) * 100

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAPE (%):", mape)

In [ ]:
tp = 40
tn = 45
fp = 5
fn = 10

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * tp / (2 * tp + fp + fn)

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall:", round(recall * 100, 2), "%")
print("F1:", round(f1 * 100, 2), "%")

In [ ]:
import numpy as np

exactitudes = np.array([0.80, 0.90, 0.85, 0.95, 1.00])

print("Media:", round(exactitudes.mean() * 100, 2), "%")
print("Desviación estándar:", round(exactitudes.std() * 100, 2), "puntos porcentuales")

In [ ]:
import numpy as np

vector = np.array([5.1, 3.5, 1.4, 0.2])
matriz = np.array([[5.1, 3.5, 1.4, 0.2]])

print("Forma del vector:", vector.shape)
print("Forma de la matriz:", matriz.shape)
print("Vector convertido a matriz:", vector.reshape(1, -1).shape)